# notebook version of python file with same name

In [1]:
import sys
from pathlib import Path

import torch

repo_root = Path("/home/jake/Documents/AI_Clinician/ai_clinician_pediatric_rl")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.rl import evaluate_terminal_reward_models as etm
# or: from src.rl.evaluate_terminal_reward_models import load_sarsa_critic

from src.rl.evaluate_terminal_reward_models import load_sarsa_critic, load_data,load_actor_and_critic, get_steps_to_go, generate_model_outputs

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# load terminal sarsa model
terminal_sarsa_critic, terminal_sarsa_config_manager, terminal_sarsa_hyperparams = load_sarsa_critic(reward_type='terminal')

# load health_score sarsa model
health_score_critic, health_score_sarsa_config_manager, health_score_sarsa_hyperparams = load_sarsa_critic(reward_type='health_score_change')

# load terminal cql model
terminal_cql_model_dir = Path("/home/jake/Documents/AI_Clinician/models/rl_models/choc/cql_with_log_probs/split_0/sepsis/intake-vis/terminal_only/final_model_terminal_discount_0.998/choc_cql_with_logprobs_intake-vis_random_search_standard_20260726_214042/unknown_cx3-20-3_20260727-131356_d435c879")
terminal_cql_actor, terminal_cql_critic, terminal_cql_config_manager, terminal_cql_hyperparams = load_actor_and_critic(model_dir = terminal_cql_model_dir, model_type='cql')

# load health_score cql model
health_score_model_dir = Path("/home/jake/Documents/AI_Clinician/models/rl_models/choc/cql_with_log_probs/split_0/sepsis/intake-vis/health_score_change_only/discount_0.975/best_runs/unknown_cx3-20-1_20260722-133922_72b7d74b")
health_score_cql_actor, health_score_cql_critic, health_score_cql_config_manager, health_score_cql_hyperparams = load_actor_and_critic(
    model_dir = health_score_model_dir, 
    model_type='cql'
)

Loading config manager from: /home/jake/Documents/AI_Clinician/models/rl_models/choc/basic_q/split_0/sepsis/intake-vis/terminal_only/final_model_terminal_discount_0.998/choc_basic_q_intake-vis_fixed_hyperparameters_standard_20260724_215318/config_manager.pkl
Detected environment: local
Checking experiments_dir: /home/jake/Documents/AI_Clinician/experiments
Ensured experiments_dir exists: /home/jake/Documents/AI_Clinician/experiments
Using supervised_final_model_dir: /home/jake/Documents/AI_Clinician/models/dynamic_deephit_results/choc/
Using bc_transformer_final_model_dir: /home/jake/Documents/AI_Clinician/models/bc_transformer/choc/
Using rl_models_dir: /home/jake/Documents/AI_Clinician/models/rl_models/choc/
Verified data_dir exists: /home/jake/Documents/AI_Clinician/choc_data
Verified code_dir exists: /home/jake/Documents/AI_Clinician/choc_code
Using config_dir: /home/jake/Documents/AI_Clinician/ai_clinician_pediatric_rl/config
Loading config manager from: /home/jake/Documents/AI_Cl

In [ ]:
processors = {}
processors['train'], processors['test'] = load_data(terminal_sarsa_config_manager, terminal_sarsa_hyperparams)  

In [ ]:
split = 'test'
processor = processors[split]
# get and create actions
clinician_actions = processor.rl_dataset["actions"]

# create zero fluid actions. We don't need zero vaso because that's what the policy outputs
zero_fluid_actions = torch.zeros_like(clinician_actions)
zero_fluid_actions[:, 1] = clinician_actions[:, 1]

# get mortality
mortality = processor.rl_dataset["mortality"]

# calculate steps to go
steps_to_go = get_steps_to_go(processor)

### calculate correlations and plot for each critic and action variant ###############
clinician_actions_np = clinician_actions.cpu().numpy() if torch.is_tensor(clinician_actions) else clinician_actions
mortality_np = mortality.cpu().numpy() if torch.is_tensor(mortality) else mortality
steps_to_go_np = steps_to_go.cpu().numpy() if torch.is_tensor(steps_to_go) else steps_to_go

dose_dim = {"zero_vaso": 1, "actor_fluid": 0, "zero_fluid": 0}  # action index per variant

# generate policy actions from the terminal cql actor
batch_size = health_score_sarsa_hyperparams['batch_size']
cql_actor_actions = generate_model_outputs(
    processor=processor, 
    batch_size=batch_size, 
    model=terminal_cql_actor,
    model_type="actor",
    device=device
)

# confirm the policy is recommending zero vaso
assert torch.all(cql_actor_actions[:, 1] == 0), "CQL actor is not recommending zero vaso for all states."

clin_fluid_actor_vaso = clinician_actions.clone()
clin_fluid_actor_vaso[:, 1] = cql_actor_actions[:, 1]
actor_fluid_clin_vaso = clinician_actions.clone()
actor_fluid_clin_vaso[:, 0] = cql_actor_actions[:, 0]

# organize the actions and critics into dictionaries for easy access
action_variants = {
    "clinician":   clinician_actions,       # baseline
    "zero_vaso":   clin_fluid_actor_vaso,  # == actor's vaso prediction, confirmed near-identical
    "actor_fluid": actor_fluid_clin_vaso,
    "zero_fluid":  zero_fluid_actions,
}

critics = {
    "cql_terminal": terminal_cql_critic,
    "sarsa_terminal": terminal_sarsa_critic,
    "sarsa_health_score": health_score_critic,
}

q_values = {}
for critic_name, critic in critics.items():
    q_values[critic_name] = {}
    for variant_name, actions in action_variants.items():
        q_values[critic_name][variant_name] = generate_model_outputs(
            processor=processor,
            batch_size=batch_size,
            model=critic,
            actions_to_use=actions if variant_name != "clinician" else None,
            model_type="critic",
            reduction='mean',
            device=device,
        )

# calculate advantages for each critic and action variant
advantages = {}
for critic_name in critics.keys():
    advantages[critic_name] = {}
    for variant_name in [k for k in action_variants.keys() if k != "clinician"]:
        advantages[critic_name][variant_name] = q_values[critic_name][variant_name] - q_values[critic_name]["clinician"]

Action std: tensor([0.2392, 0.0764], device='cuda:0')
Action std: tensor([0.2392, 0.0764], device='cuda:0')
Action std: tensor([0.2392, 0.0764], device='cuda:0')
Action std: tensor([0.2392, 0.0764], device='cuda:0')
Action std: tensor([0.2392, 0.0764], device='cuda:0')
Action std: tensor([0.2392, 0.0764], device='cuda:0')
Action std: tensor([0.2392, 0.0764], device='cuda:0')
Action std: tensor([0.2392, 0.0764], device='cuda:0')
Action std: tensor([0.2392, 0.0764], device='cuda:0')
Action std: tensor([0.2392, 0.0764], device='cuda:0')
Action std: tensor([0.2392, 0.0764], device='cuda:0')
Action std: tensor([0.2392, 0.0764], device='cuda:0')
Action std: tensor([0.2392, 0.0764], device='cuda:0')
Action std: tensor([0.2343, 0.0820], device='cuda:0')
Action std: tensor([0.2343, 0.0820], device='cuda:0')
Action std: tensor([0.2343, 0.0820], device='cuda:0')
Action std: tensor([0.2343, 0.0820], device='cuda:0')
Action std: tensor([0.2343, 0.0820], device='cuda:0')
Action std: tensor([0.2343, 

In [51]:
clinician_vaso = clinician_actions[:, 1]
# get indices of vaso dose greater than 0.1
point_one_dose_indices = torch.where(clinician_vaso > 0.1)[0]
point_one_dose_indices.shape

for critic_name, critic in critics.items():
    for variant_name, actions in action_variants.items():
        print(f"{critic_name} - {variant_name} q values mean: {q_values[critic_name][variant_name].mean()}")
        # print(f"{critic_name} - {variant_name} q values median: {q_values[critic_name][variant_name].median()}")

        q_high_vaso = q_values[critic_name][variant_name][point_one_dose_indices]
        print(f"{critic_name} - {variant_name} q values for patients with clinician vaso dose > 0.1 mean: {q_high_vaso.mean()}")

        if variant_name != "clinician":
            print(f"{critic_name} - {variant_name} advantages mean: {advantages[critic_name][variant_name].mean()}")
            # print(f"{critic_name} - {variant_name} advantages median: {advantages[critic_name][variant_name].median()}")
               
        print('\n\n')

cql_terminal - clinician q values mean: 4.825725555419922
cql_terminal - clinician q values for patients with clinician vaso dose > 0.1 mean: 1.3006294965744019



cql_terminal - zero_vaso q values mean: 4.85691499710083
cql_terminal - zero_vaso q values for patients with clinician vaso dose > 0.1 mean: 1.7157413959503174
cql_terminal - zero_vaso advantages mean: 0.031188994646072388



cql_terminal - actor_fluid q values mean: 4.944729328155518
cql_terminal - actor_fluid q values for patients with clinician vaso dose > 0.1 mean: 1.40251886844635
cql_terminal - actor_fluid advantages mean: 0.1190032809972763



cql_terminal - zero_fluid q values mean: 4.915614128112793
cql_terminal - zero_fluid q values for patients with clinician vaso dose > 0.1 mean: 1.274202823638916
cql_terminal - zero_fluid advantages mean: 0.08988804370164871



sarsa_terminal - clinician q values mean: 5.053181171417236
sarsa_terminal - clinician q values for patients with clinician vaso dose > 0.1 mean: 1.80649

In [46]:
cql_terminal_q_high_vaso = cql_terminal_q_values[point_one_dose_indices]
print(f"CQL terminal q values for patients with clinician vaso dose > 0.1 mean: {cql_terminal_q_high_vaso.mean()}")
print(f"CQL terminal q values for patients with clinician vaso dose > 0.1 median: {cql_terminal_q_high_vaso.median()}")

sarsa_terminal_q_high_vaso = sarsa_terminal_q_values[point_one_dose_indices]
print(f"SARSA terminal q values for patients with clinician vaso dose > 0.1 mean: {sarsa_terminal_q_high_vaso.mean()}")
print(f"SARSA terminal q values for patients with clinician vaso dose > 0.1 median: {sarsa_terminal_q_high_vaso.median()}")

CQL terminal q values for patients with clinician vaso dose > 0.1 mean: 1.3006294965744019
CQL terminal q values for patients with clinician vaso dose > 0.1 median: 2.6583609580993652
SARSA terminal q values for patients with clinician vaso dose > 0.1 mean: 1.8064979314804077
SARSA terminal q values for patients with clinician vaso dose > 0.1 median: 3.1821694374084473


In [47]:
cql_terminal_zero_vaso_q_high_vaso = cql_terminal_zero_vaso_q_values[point_one_dose_indices]
print(f"CQL terminal zero vaso q values for patients with clinician vaso dose > 0.1 mean: {cql_terminal_zero_vaso_q_high_vaso.mean()}")
print(f"CQL terminal zero vaso q values for patients with clinician vaso dose > 0.1 median: {cql_terminal_zero_vaso_q_high_vaso.median()}")

sarsa_terminal_zero_vaso_q_high_vaso = sarsa_terminal_zero_vaso_q_values[point_one_dose_indices]
print(f"SARSA terminal zero vaso q values for patients with clinician vaso dose > 0.1 mean: {sarsa_terminal_zero_vaso_q_high_vaso.mean()}")
print(f"SARSA terminal zero vaso q values for patients with clinician vaso dose > 0.1 median: {sarsa_terminal_zero_vaso_q_high_vaso.median()}")

CQL terminal zero vaso q values for patients with clinician vaso dose > 0.1 mean: 1.7157413959503174
CQL terminal zero vaso q values for patients with clinician vaso dose > 0.1 median: 3.0151877403259277
SARSA terminal zero vaso q values for patients with clinician vaso dose > 0.1 mean: 1.7720425128936768
SARSA terminal zero vaso q values for patients with clinician vaso dose > 0.1 median: 3.1442055702209473


In [32]:
sarsa_health_score_zero_vaso_advantage = advantages["sarsa_health_score"]["zero_vaso"]
print(f"SARSA health score zero vaso advantages mean: {sarsa_health_score_zero_vaso_advantage.mean()}")
print(f"SARSA health score zero vaso advantages median: {sarsa_health_score_zero_vaso_advantage.median()}")

SARSA health score zero vaso advantages mean: 0.02281574346125126
SARSA health score zero vaso advantages median: 0.0
